# Assignment 1: EDA and Baselines

CENG313 Introduction to Data Science, BYU - Locating Bacterial Flagellar Motors 2025.

All reusable code lives in `src/`; this notebook only calls it.

## 0. Setup

Clones the repository and adds it to the Python path. Replace `<github-user>` with the repo owner.

In [ ]:
import os, sys

REPO_URL = "https://github.com/<github-user>/GROUPxx_CENG313.git"
REPO_DIR = "/content/GROUPxx_CENG313"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

sys.path.insert(0, REPO_DIR)
!pip -q install -r {REPO_DIR}/requirements.txt

## 1. Data download

Upload your `kaggle.json` (Kaggle -> Settings -> API -> Create New Token) when prompted. Only the labels file is downloaded here; volumes are prepared on Kaggle (see README).

In [ ]:
from google.colab import files
import os

if not os.path.exists(os.path.expanduser("~/.kaggle/kaggle.json")):
    uploaded = files.upload()  # choose kaggle.json
    os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
    !mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

!mkdir -p data
!kaggle competitions download -c byu-locating-bacterial-flagellar-motors-2025 -f train_labels.csv -p data
!cd data && (unzip -o -q train_labels.csv.zip 2>/dev/null || true)
!ls -la data

In [ ]:
from src import data, utils, models, plots

labels = data.load_labels("data/train_labels.csv")
tomos = data.tomogram_table(labels)
print(labels.shape, tomos.shape)
labels.head()

## 2. Data description

_TODO: n, d, instances/features, measurement level of each column (Lecture 3)._

## 3. Metadata EDA

_TODO_

In [ ]:
plots.class_balance(tomos);

## 4. Motor position EDA

_TODO_

## 5. Image EDA

_TODO_

## 6. Baselines

In [ ]:
train_ids, val_ids = utils.split_tomograms(tomos)
train_labels = labels[labels.tomo_id.isin(train_ids)]
val_labels = labels[labels.tomo_id.isin(val_ids)]
val_tomos = tomos[tomos.tomo_id.isin(val_ids)]

results = {}
for name, model in [("B0", models.NoMotorBaseline()), ("B1", models.MeanPositionBaseline())]:
    preds = model.fit(train_labels).predict(val_tomos)
    f2, counts = utils.score(val_labels, preds, return_counts=True)
    results[name] = {"F2": round(f2, 4), **counts}

import pandas as pd
pd.DataFrame(results).T

## 7. Observations

_TODO_